# 02 · O pipeline RAG, etapa por etapa

Este notebook abre a "caixa" do `VoCRAG.ask()` e mostra, para perguntas reais, cada etapa do fluxo pedido no enunciado:

```
pergunta → processamento da pergunta → embeddings → recuperação → construção do contexto → LLM → resposta com evidências
```

e as três camadas de proteção contra respostas sem suporte (gate antes do LLM, instrução no prompt, verificação depois do LLM).

> A primeira célula imprime a configuração ativa (modelo de embeddings, reranker, LLM). Todas as saídas deste notebook correspondem a ela.

In [1]:
import os
os.environ.setdefault("HF_HUB_DISABLE_PROGRESS_BARS", "1")  # saída limpa no GitHub (sem barras de progresso)
import sys, json, textwrap
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent / "src"))
import pandas as pd
pd.set_option("display.max_colwidth", 110)

from voc_rag.pipeline import VoCRAG
from voc_rag.query import plan_query
from voc_rag.evidence import assess, build_context
from voc_rag.generation import build_messages, verify_answer
from voc_rag.retrieval import rrf_fuse

rag = VoCRAG()
s = rag.s
print(json.dumps({"embeddings": rag.index.meta["embedding_model"], "dimensao": rag.index.meta["dimensao"],
                  "documentos": rag.index.meta["n_documentos"], "reranker": s.reranker_model or "desligado",
                  "llm": f"{s.llm_provider}:{s.llm_model}", "limiares_calibrados": rag.thresholds_calibrados,
                  "limiar_relevancia": s.relevance_threshold, "limiar_escopo": s.scope_threshold,
                  "min_evidencias": s.min_evidences}, ensure_ascii=False, indent=2))

{
  "embeddings": "intfloat/multilingual-e5-base",
  "dimensao": 768,
  "documentos": 42138,
  "reranker": "cross-encoder/mmarco-mMiniLMv2-L12-H384-v1",
  "llm": "gemini:gemini-3.8-flash",
  "limiares_calibrados": true,
  "limiar_relevancia": 0.053,
  "limiar_escopo": 0.8316,
  "min_evidencias": 3
}


## 1. Processamento da pergunta
A pergunta vira um **plano**: filtros de metadados inferidos por regras explícitas (auditáveis), intenção (agregada × específica) e termos de conteúdo (usados na checagem de cobertura léxica).

In [2]:
PERGUNTA = "O que os clientes contam sobre mercadorias que chegaram quebradas ou danificadas?"
plan = plan_query(PERGUNTA, rag.known_categories)
pd.Series({"filtros": plan.filtros.describe(), "justificativas": plan.filtros_inferidos,
           "agregada?": plan.agregada, "termos de conteúdo": list(plan.termos_conteudo.values())}).to_frame("plano")

,plano
filtros,nenhum
justificativas,[]
agregada?,False
termos de conteúdo,"[mercadorias, chegaram, quebradas, danificadas]"


## 2. Representação vetorial e recuperação
Duas buscas independentes sobre o mesmo recorte:
- **densa** (embeddings + FAISS, cosseno) — encontra paráfrases;
- **léxica** (BM25 com *stemming*) — encontra termos exatos.

Os dois rankings são fundidos por **Reciprocal Rank Fusion** (RRF).

In [3]:
qv = rag.index.embedder.encode_query(PERGUNTA)
print("vetor da pergunta:", qv.shape, "| norma:", round(float((qv**2).sum()**.5), 3))
ids = plan.filtros.mask(rag.index.docs).nonzero()[0]
dense_rank, dense_scores = rag.retriever._dense(qv, ids, 5)
bm25_rank, bm25_scores = rag.retriever._bm25(PERGUNTA, plan.filtros.mask(rag.index.docs), 5)
D = rag.index.docs
show = lambda rank, sc, nome: pd.DataFrame({"posição": range(1, len(rank) + 1), nome: [round(sc[p], 3) for p in rank],
                                           "texto": D.iloc[rank]["texto"].values})
display(show(dense_rank, dense_scores, "cosseno")); display(show(bm25_rank, bm25_scores, "bm25"))

vetor da pergunta: (768,) | norma: 1.0


,posição,cosseno,texto
0,1,0.867,O produto chegou quebrado
1,2,0.867,"produto danificado. Meu produto chegou todo quebrado, quero trocar, por outro igual do mesmo"
2,3,0.866,O produto chegou quebrado.
3,4,0.865,"Boa noite recebi a mercadoria em parte pois uma cuba esta quebrada, como devo fazer? fico no prejuízo ou q..."
4,5,0.865,Produto quebrado


,posição,bm25,texto
0,1,13.523,Mercadoria quebrada. O produto chegou com a voltagem errada e quebrado. Aguardo a troca da mercadoria.
1,2,13.145,O produto chegou danificado... A película q comprei totalmente quebrada
2,3,11.750,Não chegou uma das minhas mercadorias e estar contando no site que chegou só que não chegou
3,4,11.400,"O pincel chegou quebrado, não fiquei satisfeita com a mercadoria"
4,5,11.277,o produto chegou antes do prazo.porem chegou danificado(suas rodas de plásticos vieram quebradas).


In [4]:
fused = rrf_fuse([dense_rank, bm25_rank], k=s.rrf_k)[:6]
pd.DataFrame({"doc": [D.iloc[p]["texto"][:90] for p, _ in fused], "rrf": [round(v, 4) for _, v in fused],
              "rank denso": [dense_rank.index(p) + 1 if p in dense_rank else None for p, _ in fused],
              "rank bm25": [bm25_rank.index(p) + 1 if p in bm25_rank else None for p, _ in fused]})

,doc,rrf,rank denso,rank bm25
0,O produto chegou quebrado,0.0164,1.0,NaN
1,Mercadoria quebrada. O produto chegou com a voltagem errada e quebrado. Aguardo a troca da,0.0164,NaN,1.0
2,"produto danificado. Meu produto chegou todo quebrado, quero trocar, por outro igual do mes",0.0161,2.0,NaN
3,O produto chegou danificado... A película q comprei totalmente quebrada,0.0161,NaN,2.0
4,O produto chegou quebrado.,0.0159,3.0,NaN
5,Não chegou uma das minhas mercadorias e estar contando no site que chegou só que não chego,0.0159,NaN,3.0


## 3. Seleção dos conteúdos: deduplicação + re-ranking + gate
`retrieve()` executa a recuperação completa (100 candidatos de cada busca → RRF → deduplicação de textos idênticos → re-ranking com cross-encoder). Em seguida, o **gate** decide se há evidência suficiente para chamar o LLM.

In [5]:
ret = rag.retriever.retrieve(plan)
cand = pd.DataFrame([c.to_dict() for c in ret.candidatos])
print(f"documentos no recorte: {ret.n_documentos_filtrados:,} | candidatos únicos: {len(cand)} | "
      f"reranker: {ret.usou_reranker} | score de escopo: {ret.score_escopo:.3f}")
cand[["relevancia", "score_denso", "rank_denso", "rank_bm25", "n_textos_identicos", "nota", "texto"]].head(10).round(3)

documentos no recorte: 42,138 | candidatos únicos: 40 | reranker: True | score de escopo: 0.866


,relevancia,score_denso,rank_denso,rank_bm25,n_textos_identicos,nota,texto
0,0.604,0.854,30.0,1.0,1,1,Mercadoria quebrada. O produto chegou com a voltagem errada e quebrado. Aguardo a troca da mercadoria.
1,0.294,0.850,54.0,90.0,1,1,"Entrega Danificada. Produto chegou danificado, com rachadura no corpo da fonte, entretanto funcional. Mas ..."
2,0.145,0.863,9.0,41.0,1,1,"A embalagem veio danificada assim como o produto, quebrado. Entrei em contato com a loja e ainda não tive ..."
3,0.091,0.846,87.0,27.0,1,1,recebi a mercadoria danificada e estou aguardando que o problema seja resolvido da melhor maneira possivel
4,0.071,0.867,2.0,6.0,1,1,"produto danificado. Meu produto chegou todo quebrado, quero trocar, por outro igual do mesmo"
5,0.057,0.860,11.0,NaN,1,3,"O produto veio quebrado. Um os produtos veio quebrado, estou tentando trocar ou devolver."
6,0.056,0.855,24.0,38.0,1,1,"Produto chegou quebrado, ficaram de trocar"
7,0.046,0.842,NaN,12.0,1,2,"Não recomendo. Comprei kit do lustre de diamante veio danificado e amassado , deveriam pensar mais nos cli..."
8,0.042,0.861,10.0,NaN,1,3,Produto com defeito. Produto veio quebrado
9,0.037,0.841,NaN,9.0,1,1,chegou quebrado


In [6]:
dec = assess(ret, s, plan.filtros.describe(), bool(plan.termos_conteudo), not plan.filtros.is_empty())
print("status do gate:", dec.status, "|", dec.motivo)

status do gate: ok | 7 evidências relevantes


## 4. Construção do contexto
As evidências aprovadas recebem rótulos `[E1]…[En]` e seus metadados. O LLM só pode citar esses rótulos.

In [7]:
contexto, usadas = build_context(dec.evidencias, s)
system, user = build_messages(PERGUNTA, plan.filtros.describe(), contexto)
print(user[:2500])

Pergunta: O que os clientes contam sobre mercadorias que chegaram quebradas ou danificadas?

Filtros aplicados à base: nenhum

Evidências (avaliações reais de clientes):
[E1] nota 1/5 | compra 2018-07 | categoria artes_e_artesanato | entrega atrasada 4 dia(s) | UF DF
"Mercadoria quebrada. O produto chegou com a voltagem errada e quebrado. Aguardo a troca da mercadoria."

[E2] nota 1/5 | compra 2018-08 | categoria consoles_games | entrega no prazo | UF PE
"Entrega Danificada. Produto chegou danificado, com rachadura no corpo da fonte, entretanto funcional. Mas o fato do produto chegar dessa forma, demonstra que o vendedor nao tomou os devidos cuidados no envio."

[E3] nota 1/5 | compra 2017-12 | categoria cool_stuff | entrega no prazo | UF RJ
"A embalagem veio danificada assim como o produto, quebrado. Entrei em contato com a loja e ainda não tive nenhuma resposta."

[E4] nota 1/5 | compra 2018-01 | categoria eletronicos | entrega no prazo | UF PR
"recebi a mercadoria danificada e estou

In [8]:
print(system)

Você é um analista de Voice of Customer de um e-commerce brasileiro.
Responda à pergunta do gestor usando EXCLUSIVAMENTE as avaliações de clientes fornecidas como evidências.

Regras obrigatórias:
1. Use somente o que está escrito nas evidências. Não use conhecimento externo e não suponha causas que os clientes não relataram.
2. Toda linha da resposta deve terminar com as citações das evidências que a sustentam, no formato [E1] ou [E2][E5]. Cite apenas rótulos que existem na lista.
3. Não informe números, frequências, proporções ou totais (nada de "a maioria", "muitos clientes", "60%"). Você está vendo só uma amostra recuperada por similaridade, não a base inteira.
4. Se as evidências não permitem responder à pergunta, responda apenas: SEM_EVIDENCIA_SUFICIENTE
5. Escreva em português, de forma objetiva, sem introduções.

Formato:
Resumo: <uma ou duas frases> [En]
- <tema>: <o que os clientes relatam> [En][Em]
- <tema>: <o que os clientes relatam> [En]


## 5. Geração e verificação pós-geração
A resposta crua do LLM passa por uma auditoria: citações inexistentes são removidas e **linhas sem citação são descartadas**. Se nada fundamentado sobrar, a resposta inteira é descartada.

In [9]:
raw = rag.llm.generate(system, user)
print("RESPOSTA CRUA DO LLM:\n" + raw)
texto, rep = verify_answer(raw, {e.rotulo for e in usadas})
print("\nRELATÓRIO DE VERIFICAÇÃO:"); print(json.dumps(rep.__dict__, ensure_ascii=False, indent=2))

RESPOSTA CRUA DO LLM:
Resumo: Os clientes relatam o recebimento de produtos danificados, quebrados ou rachados, apontando também problemas adicionais como embalagem avariada, voltagem incorreta e a busca por trocas, devoluções ou resolução com os vendedores [E1][E2][E3][E4][E5][E6][E7].
- Estado dos itens e embalagens: Clientes descrevem que os produtos chegaram quebrados, danificados, com a embalagem também danificada ou com rachaduras na carcaça, embora ainda funcionais [E2][E3][E5][E6][E7].
- Erros adicionais no envio: Há relato de produto que, além de quebrado, foi entregue com a voltagem errada, bem como apontamento de falta de cuidado do vendedor durante o envio [E1][E2].
- Solicitações de troca, devolução e contato: Clientes afirmam que aguardam a troca, tentam trocar ou devolver o item, esperam uma solução para o problema ou relatam falta de resposta da loja após o contato [E1][E3][E4][E5][E6][E7].

RELATÓRIO DE VERIFICAÇÃO:
{
  "citacoes_validas": [
    "E1",
    "E2",
    "E3

## 6. A mesma coisa pelo `ask()` — resposta final com evidências rastreáveis

In [10]:
from IPython.display import Markdown
r = rag.ask(PERGUNTA, modo="rag")
Markdown(r.to_markdown())

**Pergunta:** O que os clientes contam sobre mercadorias que chegaram quebradas ou danificadas?

**Status:** `ok` · **modo:** `rag`
**Filtros:** nenhum

Resumo: Clientes relatam o recebimento de produtos danificados, quebrados ou rachados, apontando também problemas adicionais no envio e buscando a resolução, troca ou devolução [E1][E2][E3][E4][E5][E6][E7].
- Condições dos itens e embalagem: Clientes relatam produtos quebrados, rachaduras no corpo do item e embalagem também danificada, mencionando inclusive a falta de cuidado do vendedor no envio [E2][E3][E5][E6].
- Divergências adicionais: Além de quebrado, houve caso de recebimento do produto com a voltagem errada [E1].
- Solicitação de troca e devolução: Os clientes informam que aguardam, tentam realizar ou receberam a promessa de troca ou devolução dos itens quebrados [E1][E5][E6][E7].
- Contato e resolução com a loja: Relatos apontam clientes aguardando uma solução para a mercadoria danificada e ausência de resposta da loja após o contato [E3][E4].

> Base da resposta: 7 avaliações recuperadas por similaridade semântica e léxica. É uma amostra das avaliações mais relevantes, não uma contagem da base.

| rótulo | review_id | nota | compra | categoria | entrega | relevância | citada | texto |
|---|---|---|---|---|---|---|---|---|
| E1 | `ea53de1722fff5127f7a879e308b1b01` | 1 | 2018-07 | artes_e_artesanato | atrasada | 0.604 | sim | Mercadoria quebrada. O produto chegou com a voltagem errada e quebrado. Aguardo a troca da mercadoria. |
| E2 | `aa06519a57b802f420fd54410ad60613` | 1 | 2018-08 | consoles_games | no_prazo | 0.294 | sim | Entrega Danificada. Produto chegou danificado, com rachadura no corpo da fonte, entretanto funcional. Mas o fato do produto chegar dessa forma, demonstra que o vendedor nao tomou os devidos cuidados no envio. |
| E3 | `6afb9abc15ff32b17abc8f3f819ade1f` | 1 | 2017-12 | cool_stuff | no_prazo | 0.145 | sim | A embalagem veio danificada assim como o produto, quebrado. Entrei em contato com a loja e ainda não tive nenhuma resposta. |
| E4 | `6245c9a6fbd78013fb3c7128c1df45bb` | 1 | 2018-01 | eletronicos | no_prazo | 0.091 | sim | recebi a mercadoria danificada e estou aguardando que o problema seja resolvido da melhor maneira possivel |
| E5 | `ac8881d2f6663930894ce0e7dbc711c8` | 1 | 2018-08 | sem_itens | nao_entregue | 0.071 | sim | produto danificado. Meu produto chegou todo quebrado, quero trocar, por outro igual do mesmo |
| E6 | `206fcd0c7e9d3248ff891f31205ae6ee` | 3 | 2018-08 | moveis_decoracao | no_prazo | 0.057 | sim | O produto veio quebrado. Um os produtos veio quebrado, estou tentando trocar ou devolver. |
| E7 | `ea13f408f9f91e78274f1caa2c9663d2` | 1 | 2017-07 | moveis_decoracao | no_prazo | 0.056 | sim | Produto chegou quebrado, ficaram de trocar |

## 7. Metadados: filtros inferidos da pergunta

In [11]:
r = rag.ask("O que dizem os clientes de SP sobre pedidos entregues com atraso?", modo="rag")
print("filtros:", r.plano["filtros_descricao"], "| documentos no recorte:", r.diagnostico["n_documentos_filtrados"])
Markdown(r.to_markdown())

filtros: UF ∈ {SP}; entrega ∈ {atrasada} | documentos no recorte: 999


**Pergunta:** O que dizem os clientes de SP sobre pedidos entregues com atraso?

**Status:** `ok` · **modo:** `rag`
**Filtros:** UF ∈ {SP}; entrega ∈ {atrasada}

Resumo: Os clientes de SP relatam o descumprimento do prazo de entrega, a falta de avisos ou esclarecimentos sobre o atraso e não recomendam a compra, mencionando também a intenção de cancelamento caso o pedido não chegue [E1][E2][E3][E4][E5][E6][E7][E8].
- Falta de comunicação: Os clientes reclamam que não foram avisados sobre o atraso e demandam esclarecimentos sobre o motivo de não terem recebido a entrega [E3][E4][E5].
- Não recomendação e cancelamento: Há relatos desaconselhando a compra devido ao atraso e menção de cancelar o pedido se a entrega não ocorrer logo [E2][E4].
- Atraso na entrega e espera: Clientes registram que os prazos não foram cumpridos, apontam atraso na entrega e relatam ainda estar aguardando a chegada dos produtos [E1][E4][E5][E6][E7][E8].

> Base da resposta: 8 avaliações recuperadas por similaridade semântica e léxica. É uma amostra das avaliações mais relevantes, não uma contagem da base.

| rótulo | review_id | nota | compra | categoria | entrega | relevância | citada | texto |
|---|---|---|---|---|---|---|---|---|
| E1 | `364e74a2d89a35fa3fad2beaf96d8f17` | 1 | 2018-07 | moveis_escritorio | atrasada | 0.237 | sim | Aguardando a entrega. Entrega em atraso. |
| E2 | `108195606e37780c4335f0725c4c7e3e` | 2 | 2018-08 | automotivo | atrasada | 0.133 | sim | Não recomendado. Atraso na entrega. |
| E3 | `f6113199cd352db89c5a7e59b903d549` | 1 | 2018-03 | bebes | atrasada | 0.123 | sim | Entrega super em atraso, preciso de um esclarecimento do ocorrido |
| E4 | `30fe371caf618181a6a5180d076e83fc` | 1 | 2018-07 | telefonia | atrasada | 0.090 | sim | Não recomendo. Atraso na entrega, ainda não recebi e não sou informado o por que do atraso.. Se não receber logo vou cancelar o pedido |
| E5 | `85a391b84d8e864de7190a2afb6f82e4` | 1 | 2018-08 | alimentos | atrasada | 0.085 | sim | não cumpre prazo. Não cumpriram prazo de entrega e nem avisaram o atraso. |
| E6 | `44dba35181b574d4779194f1c8e43e20` | 1 | 2018-02 | automotivo | atrasada | 0.085 | sim | Atraso na entrega. |
| E7 | `9f5600c5ab1457f1ec08b2ea6720cf49` | 1 | 2017-10 | moveis_escritorio | atrasada | 0.073 | sim | entrega muito atrasada. |
| E8 | `5193565d8ef972341e963399fedec040` | 1 | 2018-01 | cama_mesa_banho | atrasada | 0.073 | sim | Entrega atrasada |

## 8. Quando NÃO responder
Três situações exigidas pelo enunciado (requisito 4), cada uma barrada por uma camada diferente do gate:

| Situação | Pergunta | Camada que barra |
|---|---|---|
| Fora do escopo | "Qual é a capital da Austrália?" | score de escopo (similaridade com a base) |
| Sem avaliações relevantes | "Quais reclamações existem sobre o programa de cashback?" | cobertura léxica: o termo não aparece em nenhuma avaliação |
| Evidências insuficientes | "O que os clientes relatam sobre a compra de patinetes?" | termo raro: aparece em 1 avaliação, que é mostrada sem generalização |

A tabela abaixo é a saída real: a coluna "motivo" mostra qual checagem decidiu.


In [12]:
casos = ["Qual é a capital da Austrália?", "Quais reclamações existem sobre o programa de cashback?",
         "O que os clientes relatam sobre a compra de patinetes?"]
linhas = []
for q in casos:
    r = rag.ask(q)
    linhas.append({"pergunta": q, "status": r.status, "motivo": r.diagnostico.get("motivo_gate"),
                   "mensagem ao usuário": r.avisos[0] if r.avisos else "", "evidências exibidas": len(r.evidencias)})
pd.DataFrame(linhas)

,pergunta,status,motivo,mensagem ao usuário,evidências exibidas
0,Qual é a capital da Austrália?,fora_do_escopo,score de escopo 0.780 < limiar 0.832; termos sem nenhuma ocorrência na base: 'australia',"A pergunta parece estar fora do escopo da base de conhecimento, que contém apenas avaliações de clientes d...",0
1,Quais reclamações existem sobre o programa de cashback?,sem_evidencias,termos sem nenhuma ocorrência na base: 'cashback',Não encontrei avaliações relevantes para responder a esta pergunta na base de conhecimento (termos sem nen...,0
2,O que os clientes relatam sobre a compra de patinetes?,evidencias_insuficientes,termos raros na base: 'patinetes' (1 avaliação(ões)),Encontrei apenas 1 avaliação(ões) relevante(s) (termos raros na base: 'patinetes' (1 avaliação(ões))). É p...,1


## 9. Pergunta agregada → camada analítica
"Principais problemas" pede **frequência**. O modo automático encaminha a pergunta para a camada analítica: as contagens vêm do código (taxonomia de temas sobre a base inteira) e o LLM apenas redige a síntese, citando exemplos e copiando números da tabela — linhas com números fora da tabela são removidas.

Antes de contar, a pergunta passa pelas mesmas barreiras do gate (termo ausente, escopo, termo raro); se o recorte tiver menos de 30 avaliações com comentário, o sistema não calcula porcentagens.


In [13]:
r = rag.ask("Quais são os principais problemas relatados pelos clientes em suas avaliações?")
print("modo:", r.modo, "| status:", r.status)
display(pd.DataFrame(r.estatisticas["tabela"])[["tema", "avaliacoes", "pct_base", "nota_media"]])
print("verificação:", json.dumps({k: v for k, v in r.diagnostico.get("verificacao", {}).items()}, ensure_ascii=False)[:800])
Markdown(r.to_markdown())

modo: analitico | status: ok


,tema,avaliacoes,pct_base,nota_media
0,Pedido não recebido,3406,"23,6%",1.29
1,Não recomenda / insatisfação geral,1710,"11,8%",1.33
2,"Troca, devolução, cancelamento ou reembolso",1704,"11,8%",1.35
3,Pedido incompleto / faltando itens,1484,"10,3%",1.54
4,Atraso na entrega,1215,"8,4%",1.66


verificação: {"citacoes_validas": ["E1", "E2", "E3", "E4", "E5", "E6", "E7", "E8", "E9", "E10"], "citacoes_invalidas": [], "linhas_removidas_sem_citacao": [], "expressoes_quantitativas": [], "llm_declarou_sem_evidencia": false, "numeros_fora_da_tabela": []}


**Pergunta:** Quais são os principais problemas relatados pelos clientes em suas avaliações?

**Status:** `ok` · **modo:** `analitico`
**Filtros:** nenhum

Resumo: Os principais problemas relatados são pedido não recebido (3406 avaliações, 23,6%), não recomenda / insatisfação geral (1710 avaliações, 11,8%) e solicitações de troca, devolução, cancelamento ou reembolso (1704 avaliações, 11,8%).
- Pedido não recebido (3406 avaliações, 23,6%): Clientes relatam que a compra ainda não chegou, que seguem esperando a entrega e apontam descaso pela não entrega do produto [E1][E2].
- Não recomenda / insatisfação geral (1710 avaliações, 11,8%): Clientes manifestam indignação com o descumprimento de prazos, falta de profissionalismo, ausência de entrega e afirmam que é impossível recomendar o serviço [E3][E4].
- Troca, devolução, cancelamento ou reembolso (1704 avaliações, 11,8%): Clientes solicitam reembolso após problemas na entrega ou pedem troca/reembolso devido ao recebimento de produto com defeito [E5][E6].
- Pedido incompleto / faltando itens (1484 avaliações, 10,3%): Clientes apontam o recebimento de apenas parte da compra, cobrando a entrega dos itens restantes que faltaram [E7][E8].
- Atraso na entrega (1215 avaliações, 8,4%): Clientes reclamam da demora e de compras que ultrapassaram o prazo previsto de entrega, além da falta de tratamento do problema [E9][E10].

> Contagens calculadas sobre a base (98410 avaliações no recorte (42138 com comentário); filtros: nenhum).
> Método: taxonomia de temas por regras léxicas auditáveis; uma avaliação pode ter mais de um tema. Cobertura da taxonomia (problema): 68,2% das 14445 avaliações da base de contagem.

| rótulo | review_id | nota | compra | categoria | entrega | relevância | citada | texto |
|---|---|---|---|---|---|---|---|---|
| E1 | `20787e5ce2d5175a20135e52e3a701e8` | 2 | 2018-05 | sem_itens | nao_entregue | 0.713 | sim | Problemas com entrega. Não chegou ainda estou na esperando ainda |
| E2 | `4c2ab64ffc9119884d87d90bf9d630c3` | 1 | 2017-03 | sem_itens | nao_entregue | 0.561 | sim | Descaso com o cliente. O produto não foi entregue. |
| E3 | `b40a6984937cc251558d24d0b5caa929` | 1 | 2018-07 | beleza_saude | atrasada | 0.462 | sim | não recomendo. Não cumprem os ´prazos e ignoram a expectativa do cliente. Falta profissionalismo. Impossível recomendar..... |
| E4 | `7c8154d29d239b02ad0e8f2dcdf3261a` | 1 | 2017-12 | relogios_presentes | atrasada | 0.121 | sim | Péssimo o procedimento. Vergonha. Nada entregue até agora. Absurdo e descaso com cliente. |
| E5 | `2e6ec612df2ad503c839b73a704566cf` | 1 | 2018-03 | papelaria | atrasada | 0.205 | sim | Ouve um problema e a entrega não foi realizada quero o reembolso |
| E6 | `1750446f4afa39b6cb4905d0ff292c3d` | 3 | 2018-03 | telefonia | no_prazo | 0.154 | sim | Produto chegou no prazo, mas com defeito. Estou pedindo troca ou reembolso. |
| E7 | `84775964cfaf2e4fa80f8657d4cdede8` | 1 | 2018-06 | cine_foto | no_prazo | 0.826 | sim | Problemas. Entregaram apenas um produto dos quatro itens do pedido... Quero o restante do pedido... Cadê? |
| E8 | `7d81592881b86d6ba09e4e5ae954a59d` | 1 | 2017-12 | ferramentas_jardim | no_prazo | 0.639 | sim | Chegou o produto atrasado, faltando outro. Falta chegar os demais. O cliente está zangado. |
| E9 | `1da9941264ebc5c4a093ee4ba82af918` | 1 | 2017-11 | brinquedos | no_prazo | 0.108 | sim | O produto chegou correto. O problema foi a demora na entrega. |
| E10 | `4baf65ba0f4dc753e9cf3d8863433da0` | 2 | 2018-01 | papelaria | atrasada | 0.064 | sim | Falta de consideração com o cliente, pois ocorreu problema na entrega é em momento nenhum o problema foi tratado.duas compras, uma com prazo de entrega 15 e 21/02, as 2 atrasou, liguei dia 23 reclaman |